# 筛选apparent temperature > 33 ℃的数据

In [ ]:
import glob
import os
import pandas as pd
input_dir = r'D:\seoul\a_airtem'
csv_file = os.path.join(input_dir, 'AT_2015_2024.csv')
df = pd.read_csv(csv_file, skiprows=3, encoding='euc_kr')

df

In [ ]:
print(df)

In [ ]:
import pandas as pd

# 1. 读取并解析日期
path = r'D:\seoul\a_airtem\AT_2015_2024.csv'
df = pd.read_csv(path, skiprows=3, encoding='euc_kr', parse_dates=['일자'])

# 2. 按日期排序
df = df.sort_values('일자')
print(df)
# 3. 标记“体感温度 > 33”
df['is_hot'] = df['체감온도(°C)'] > 33

# 4. 计算前一天记录与当前行的日期差
df['prev_diff'] = df['일자'] - df['일자'].shift(1)

# 5. 标记“与前一天连续且两天都 >33”
df['two_day_consec'] = (
    df['is_hot'] &
    df['is_hot'].shift(1) &
    (df['prev_diff'] == pd.Timedelta(days=1))
)

# 6. 把“连续两天”中的两个日期都选出来
mask = df['two_day_consec'] | df['two_day_consec'].shift(-1).fillna(False)
result = df.loc[mask, ['일자', '기온(°C)', '습도(%rh)', '체감온도(°C)']]

# 7. 输出
print("=== 连续两天体感温度 > 33°C 的记录 ===")
print(result.to_string(index=False))
result.to_csv(r'D:\seoul\a_airtem\result\highest_temps_over_33.csv')

In [ ]:
import pandas as pd

path = r'D:\seoul\a_airtem\result\highest_temps_over_33.csv'

# 用和写出时一样的编码来读取
df = pd.read_csv(path, encoding='utf-8-sig', parse_dates=['일자'])

# 现在就能打印 일자 列了
print(df['일자'])

In [ ]:
import numpy as np

unique_dates = np.array(df['일자'].dt.strftime('%Y-%m-%d'))
print(unique_dates)

# 02 需要下载全部的对应文件的 B10 和 Qixel 处理 extreme heat date
 https://earthexplorer.usgs.gov/

In [ ]:
landsat_8 = r'D:\seoul\b_satellite_img\landsat_ot_c2_l2.csv'
df = pd.read_csv(landsat_8, encoding='ISO-8859-1')
df

In [ ]:
# 过滤只保留 Collection Category 为 T1 的数据
df_t1 = df[df['Collection Category'] == 'T1']
df_t1['Date Acquired']

In [ ]:
unique_dates = unique_dates
acquired    = df_t1['Date Acquired'].to_numpy()
#print(unique_dates)
#print(acquired)
# 打印类型和元素的数据类型（如果有 dtype 属性）
print("unique_dates →", type(unique_dates), getattr(unique_dates, 'dtype', None))
print("acquired    →", type(acquired), acquired.dtype)

In [ ]:
import pandas as pd

unique_dates = unique_dates
acquired    = pd.to_datetime(
    df_t1['Date Acquired'],
    format='%Y/%m/%d',  # 注意这里是“/”
    errors='coerce'
).values
unique_dates_clean = unique_dates.astype('datetime64[D]')
acquired_clean      = acquired.astype   ('datetime64[D]')

In [ ]:
import pandas as pd

# 求交集
common = set(acquired_clean) & set(unique_dates_clean)
# 把 set 转成列表并排序
common_sorted = sorted(common)
# （如果需要字符串格式，可以再 map 一次 str）
common_str = [str(d) for d in common_sorted]
print(f"共有 {len(common_str)} 个相同的日期：{common_str}")
for d in common_str:
    print(d)

In [ ]:
sorted_dates = [str(d).replace("-", "") for d in common_sorted]
print(sorted_dates)  # 输出: 20240618

# 然后下载对应的文件

In [ ]:
import os
import rasterio
import geopandas as gpd
from rasterio.mask import mask
from glob import glob

def clip_rasters_with_shapefile(raster_folder, shapefile_path, output_folder):
    # Ensure the output directory exists
    os.makedirs(output_folder, exist_ok=True)

    # Load the shapefile as a GeoDataFrame
    nyc_shape = gpd.read_file(shapefile_path)

    # Loop through each raster in the raster folder
    for raster_path in glob(os.path.join(raster_folder, '*.tif')):
        with rasterio.open(raster_path) as src:
            raster_crs = src.crs

            # Ensure the shapefile is in the same CRS as the raster
            if nyc_shape.crs != raster_crs:
                nyc_shape = nyc_shape.to_crs(raster_crs)

            # Clip the raster using the shapefile
            out_image, out_transform = mask(src, nyc_shape.geometry, crop=True)

            # Update metadata for the output raster
            out_meta = src.meta.copy()
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "crs": raster_crs  # Use the raster's CRS for the output
            })

            # Save the clipped raster to the output folder
            output_raster_path = os.path.join(output_folder, f"clipped_{os.path.basename(raster_path)}")
            if os.path.isdir(output_raster_path):
                continue
            with rasterio.open(output_raster_path, "w", **out_meta) as dest:
                dest.write(out_image)

            print(f"Clipped raster saved to: {output_raster_path}")

In [ ]:
# Paths to the folder and shapefile
raster_folder = r'D:\seoul\b_satellite_img\extreme\original'
seoul_boundary_file = r'D:\seoul\Final_data\Admin_boundary\Seoul_boundary.shp'
shp_seoul = seoul_boundary_file
output_folder = r'D:\seoul\b_satellite_img\extreme\clipped'
os.makedirs(output_folder, exist_ok=True)

clip_rasters_with_shapefile(raster_folder, shp_seoul, output_folder)

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import glob
# Mapping from month names to numbers for date comparison
month_to_num = {
    'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04', 'May': '05', 'Jun': '06',
    'Jul': '07', 'Aug': '08', 'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'
}

def calculate_pixel_proportion(sorted_dates, base_folder, pixel_values=[21824, 21952]):
    for sorted_data in sorted_dates:
        pattern = r'{}\*_*{}_*_QA_PIXEL.TIF'.format(base_folder, sorted_data)
        matched_files = glob.glob(pattern)
        if not matched_files:
            continue
        with rasterio.open(matched_files[0]) as src:
            qa_pixel_data = src.read(1)
        non_zero_pixels = qa_pixel_data[qa_pixel_data > 0]
        total_non_zero_pixels = non_zero_pixels.size
        unique, counts = np.unique(non_zero_pixels, return_counts=True)
        pixel_counts = dict(zip(unique, counts))
        pixel_proportions = {value: count / total_non_zero_pixels for value, count in pixel_counts.items()}

        print(f"📅 {sorted_data}:")
        val_all = 0
        for val in pixel_values:
            prop = pixel_proportions.get(val, 0)
            print(f"  - 像素值 {val} 的占比: {prop:.2%}")
            val_all = val_all + prop
        print(val_all)


# Example usage
base_folder = r'D:\seoul\b_satellite_img\extreme\clipped'

calculate_pixel_proportion(sorted_dates, base_folder)

# extreme heat day:
finally we choose the 20230819(0.726) and 20160807(0.825)
# normal heat day:
then I need to check the normal heat day percentile (40-60%) during the recent 30 years （1995-2024）

In [ ]:
import os
import pandas as pd

input_dir = r'D:\seoul\a_airtem'
csv_files = [
    os.path.join(input_dir, 'AT_1995_2004.csv'),
    os.path.join(input_dir, 'AT_2005_2014.csv'),
    os.path.join(input_dir, 'AT_2015_2024.csv'),
]

# 读取并合并
dfs = [pd.read_csv(f, skiprows=3, encoding='euc_kr') for f in csv_files]
tem = pd.concat(dfs, ignore_index=True)

# 查看合并结果
print(tem.head())
tem.to_csv(r'D:\seoul\a_airtem\result\tem.csv')

In [ ]:
file = r'D:\seoul\a_airtem\result\tem.csv'
data = pd.read_csv(file)
data
# 计算每行 AT(체감온도(°C)) 的百分等级（0~100）
data['percentile'] = data['체감온도(°C)'].rank(pct=True) * 100

# 只保留你关心的几列
out_cols = ['일자', '기온(°C)', '습도(%rh)', '체감온도(°C)', 'percentile']
data_out = data[out_cols]

# 6. 保存为 UTF-8-SIG 编码的 CSV
out_fp = r'D:\seoul\a_airtem\result\temps.csv'
data_out.to_csv(out_fp, index=False, encoding='utf-8-sig')
print(f"✔ 已保存 {len(data_out)} 条 5/6/7/8/9 月记录（含 percentile 列）到：{out_fp}")


In [ ]:
file = r'D:\seoul\a_airtem\result\temps.csv'
data = pd.read_csv(file)
data

In [ ]:
file = r'D:\seoul\a_airtem\result\temps.csv'
data = pd.read_csv(file)
data

# 条件筛选：百分位在 40 到 60 之间的行
filtered_data = data[(data['percentile'] >= 40) & (data['percentile'] <= 60)]

# 打印这些行的 '일시' 列
print(filtered_data[['일자', 'percentile']])


In [ ]:
# 转换日期格式：去掉 "-"，筛选以 2024 或 2016 开头的日期
sorted_dates = [str(d).replace("-", "") for d in filtered_data['일자']]
filtered_dates = [d for d in sorted_dates if d.startswith('2023') or d.startswith('2016')]

# 用 boolean mask 获取对应的行（原始 '일시' 格式）
filtered_rows = filtered_data[filtered_data['일자'].astype(str).str.replace("-", "").isin(filtered_dates)]

# 打印日期和 percentile 两列
filtered_rows[['일자', 'percentile']]


In [ ]:
import numpy as np
import pandas as pd

# 将 '일시' 列转换为字符串形式的日期
normal_dates_clean = pd.to_datetime(filtered_data['일자']).dt.strftime('%Y-%m-%d')
filtered_dates = [d for d in normal_dates_clean if d.startswith('2023') or d.startswith('2016')]

set_normal_array = np.array(filtered_dates)

# 将 acquired_clean 转为字符串形式
acquired_str_array = np.array([str(d)[:10] for d in acquired_clean])  # 保留到日，防止有时间部分

# 取交集
common = np.intersect1d(acquired_str_array, set_normal_array)

# 排序并转成列表
common_sorted = sorted(common.tolist())
print(f"共有 {len(common_sorted)} 个相同的日期：{common_sorted}")


In [ ]:
import os
import rasterio
import geopandas as gpd
from rasterio.mask import mask
from glob import glob

def clip_rasters_with_shapefile(raster_folder, shapefile_path, output_folder):
    # Ensure the output directory exists
    os.makedirs(output_folder, exist_ok=True)

    # Load the shapefile as a GeoDataFrame
    nyc_shape = gpd.read_file(shapefile_path)

    # Loop through each raster in the raster folder
    for raster_path in glob(os.path.join(raster_folder, '*.tif')):
        with rasterio.open(raster_path) as src:
            raster_crs = src.crs

            # Ensure the shapefile is in the same CRS as the raster
            if nyc_shape.crs != raster_crs:
                nyc_shape = nyc_shape.to_crs(raster_crs)

            # Clip the raster using the shapefile
            out_image, out_transform = mask(src, nyc_shape.geometry, crop=True)

            # Update metadata for the output raster
            out_meta = src.meta.copy()
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "crs": raster_crs  # Use the raster's CRS for the output
            })

            # Save the clipped raster to the output folder
            output_raster_path = os.path.join(output_folder, f"clipped_{os.path.basename(raster_path)}")
            if os.path.isdir(output_raster_path):
                continue
            with rasterio.open(output_raster_path, "w", **out_meta) as dest:
                dest.write(out_image)

            print(f"Clipped raster saved to: {output_raster_path}")

In [ ]:
# Paths to the folder and shapefile
raster_folder = r'D:\seoul\b_satellite_img\normal\original'
seoul_boundary_file = r'D:\seoul\Final_data\Admin_boundary\Seoul_boundary.shp'
shp_seoul = seoul_boundary_file
output_folder = r'D:\seoul\b_satellite_img\normal\clipped'
os.makedirs(output_folder, exist_ok=True)

clip_rasters_with_shapefile(raster_folder, shp_seoul, output_folder)

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import glob
# Mapping from month names to numbers for date comparison
month_to_num = {
    'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04', 'May': '05', 'Jun': '06',
    'Jul': '07', 'Aug': '08', 'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'
}

def calculate_pixel_proportion(sorted_dates, base_folder, pixel_values=[21824, 21952]):
    for sorted_data in sorted_dates:
        pattern = r'{}\*_*{}_*_QA_PIXEL.TIF'.format(base_folder, sorted_data)
        matched_files = glob.glob(pattern)
        if not matched_files:
            continue
        with rasterio.open(matched_files[0]) as src:
            qa_pixel_data = src.read(1)
        non_zero_pixels = qa_pixel_data[qa_pixel_data > 0]
        total_non_zero_pixels = non_zero_pixels.size
        unique, counts = np.unique(non_zero_pixels, return_counts=True)
        pixel_counts = dict(zip(unique, counts))
        pixel_proportions = {value: count / total_non_zero_pixels for value, count in pixel_counts.items()}

        print(f"📅 {sorted_data}:")
        val_all = 0
        for val in pixel_values:
            prop = pixel_proportions.get(val, 0)
            print(f"  - 像素值 {val} 的占比: {prop:.2%}")
            val_all = val_all + prop
        print(val_all)


# Example usage
base_folder = r'D:\seoul\b_satellite_img\normal\clipped'

calculate_pixel_proportion(sorted_dates, base_folder)

# 确定了 extreme heat and normal heat days
extreme heat 20160807, 20230819
normal heat 20160924, 20230616

# 下一步是得到 heat resilience map

# 用 QA数据清理B10 -> clean data

In [ ]:
import rasterio
import numpy as np
import os
import glob

extreme_heat=[20160807, 20230819]
normal_heat=[20160924, 20230616]
label_dict = {d: 'extreme' for d in extreme_heat}
label_dict.update({d: 'normal' for d in normal_heat})

for date, label in label_dict.items():
    print(f"{label} heat day:", date)
    data_workspace = rf"D:\seoul\b_satellite_img\{label}\clipped"
    b_10_data = glob.glob(os.path.join(data_workspace, rf'clipped_*_{date}_*_ST_B10.TIF'))
    qa_data = glob.glob(os.path.join(data_workspace, rf'clipped_*_{date}_*_QA_PIXEL.TIF'))
    output_folder = os.path.join(data_workspace, r'clean')
    os.makedirs(output_folder, exist_ok=True)

    if not b_10_data or not qa_data:
        print(f"No file found for date {date}")
        continue

    b_10_data = b_10_data[0]
    qa_data = qa_data[0]

    output_data = os.path.join(output_folder, rf'clipped_{date}_B10_cleaned.tif')
    print("B10 Data Path:", b_10_data)
    print("QA Data Path:", qa_data)
    print("Output Data Path:", output_data)

    # Read Band 10 and QA data
    with rasterio.open(b_10_data) as src:
        band10 = src.read(1).astype(float)
        profile = src.profile

    with rasterio.open(qa_data) as qa_src:
        qa = qa_src.read(1)

    # QA filtering
    valid_labels = [21824, 21952]
    cloud_mask = ~np.isin(qa, valid_labels)
    band10[cloud_mask] = 0

    with rasterio.open(output_data, 'w', **profile) as dst:
        dst.write(band10, 1)



In [ ]:
import os
import rasterio
import geopandas as gpd
from rasterio.features import shapes
from shapely.geometry import shape

def raster_to_shapefile(raster_files, output_folder):
    # 确保输出目录存在
    os.makedirs(output_folder, exist_ok=True)

    for raster_file in raster_files:
        with rasterio.open(raster_file) as src:
            # 获取栅格的nodata值
            nodata_value = src.nodata
            # 输出 Shapefile 文件路径
            output_shapefile = os.path.join(output_folder, os.path.basename(raster_file).replace('.tif', '.shp'))
            if os.path.exists(output_shapefile):
                continue
            # 保存为 Shapefile
            gdf.to_file(output_shapefile)

            # 提取栅格的形状
            mask = src.read(1)  # 读取第一个波段的数据
            results = shapes(mask, mask=mask != nodata_value, transform=src.transform)

            # 将结果转换为 GeoDataFrame
            shapes_list = []
            for geom, value in results:
                geom = shape(geom)
                if geom.is_valid:
                    shapes_list.append({"geometry": geom, "value": value})

            # 创建 GeoDataFrame
            gdf = gpd.GeoDataFrame(shapes_list, crs=src.crs)


            print(f"栅格已转换为 Shapefile 并保存到: {output_shapefile}")
extreme_heat=[20160807, 20230819]
normal_heat=[20160924, 20230616]
# 输入的栅格文件路径
normal_files = [
    rf'D:\seoul\b_satellite_img\\normal\clipped\clean\clipped_{normal_heat[0]}_B10_cleaned.tif',
    rf'D:\seoul\b_satellite_img\\normal\clipped\clean\clipped_{normal_heat[1]}_B10_cleaned.tif'
]
extreme_files = [
    rf'D:\seoul\b_satellite_img\extreme\clipped\clean\clipped_{extreme_heat[0]}_B10_cleaned.tif',
    rf'D:\seoul\b_satellite_img\extreme\clipped\clean\clipped_{extreme_heat[1]}_B10_cleaned.tif'
]

# 合并所有栅格文件
all_raster_files = normal_files + extreme_files

# 输出文件夹路径
output_shapefile_folder = r'D:\seoul\b_satellite_img\shapefiles'
os.makedirs(output_shapefile_folder, exist_ok=True)

# 转换栅格文件为 Shapefile
raster_to_shapefile(all_raster_files, output_shapefile_folder)

# 得到fid的shape文件

In [ ]:
import geopandas as gpd
import pandas as pd
import os
import glob

def fast_clip_and_assign_fid(input_shp, clip_shp, output_shp):
    print(f"📂 加载输入数据：{input_shp}")
    input_gdf = gpd.read_file(input_shp)
    clip_gdf = gpd.read_file(clip_shp)

    # 确保 CRS 一致
    if input_gdf.crs != clip_gdf.crs:
        input_gdf = input_gdf.to_crs(clip_gdf.crs)

    # 在裁剪前先加上 city_id 字段
    clip_gdf = clip_gdf.reset_index().rename(columns={'index': 'city_id'})

    print("✂️ 正在执行 overlay 精准裁剪...")
    try:
        clipped = gpd.overlay(input_gdf, clip_gdf[['city_id', 'geometry']], how='intersection', keep_geom_type=False)
    except Exception as e:
        print(f"❌ Overlay 错误: {e}")
        return

    # 保存结果
    os.makedirs(os.path.dirname(output_shp), exist_ok=True)
    clipped.to_file(output_shp, driver='ESRI Shapefile')
    print(f"✅ 已保存: {output_shp}")
extreme_heat=[20160807, 20230819]
normal_heat=[20160924, 20230616]

# 输入的栅格转 Shapefile 路径（假设已经将其转化为 Shapefile）
shapefile_files = [
    rf'D:\seoul\b_satellite_img\shapefiles\clipped_{normal_heat[0]}_B10_cleaned.shp',
    rf'D:\seoul\b_satellite_img\shapefiles\clipped_{normal_heat[1]}_B10_cleaned.shp',
    rf'D:\seoul\b_satellite_img\shapefiles\clipped_{extreme_heat[0]}_B10_cleaned.shp',
    rf'D:\seoul\b_satellite_img\shapefiles\clipped_{extreme_heat[1]}_B10_cleaned.shp'

]
grid_folder = r'D:\seoul\grids'
# 输出文件夹路径
output_folder = r'D:\seoul\b_satellite_img\shapefiles_clipped_with_fid'
# 遍历所有 grid shapefile
for grid_file in os.listdir(grid_folder):
    if grid_file.endswith('.shp'):
        grid_path = os.path.join(grid_folder, grid_file)

        for input_shp in shapefile_files:
            input_name = os.path.splitext(os.path.basename(input_shp))[0]
            grid_name = os.path.splitext(os.path.basename(grid_file))[0]
            output_name = f'{input_name}_clipped_by_{grid_name}_with_fid.shp'
            output_shp = os.path.join(output_folder, output_name)
            if os.path.exists(output_shp):
                print(f"⏩ 已存在，跳过: {output_shp}")
                continue
            fast_clip_and_assign_fid(input_shp, grid_path, output_shp)

# 先将数据洗成摄氏度 bt -> lst

In [2]:
import rasterio
import numpy as np
import os


def calculate_lst(input_files, output_dir, multiplier=0.00341802, add_constant=149.0, nodata_value=-9999):
    """
    Calculate Land Surface Temperature (LST) from a list of raster files and save the output.

    Args:
        input_files (list): List of paths to input raster files.
        output_dir (str): Directory to save the output raster files.
        multiplier (float): Multiplication factor for converting to temperature.
        add_constant (float): Additive constant for converting to temperature.
        nodata_value (float): Value to use for no-data in the output files.
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    for input_file in input_files:
        with rasterio.open(input_file) as src:
            # Read the first band and convert to float
            band_data = src.read(1).astype(float)
            profile = src.profile

        # Replace values <= 0 with NaN
        band_data[band_data <= 0] = np.nan

        # Calculate brightness temperature (BT)
        lst_kelvin = (band_data * multiplier) + add_constant

        # Convert to Celsius
        lst_celsius = lst_kelvin - 273.15

        # Replace NaN values with the nodata value
        lst_celsius[np.isnan(lst_celsius)] = nodata_value

        # Update profile for the output file
        profile.update({
            'dtype': 'float32',
            'count': 1,
            'nodata': nodata_value
        })

        # Generate a short output file name
        base_name = os.path.basename(input_file).split('.')[0]
        output_file = os.path.join(output_dir, f"{base_name}_LST.tif")

        # Save the LST raster
        with rasterio.open(output_file, 'w', **profile) as dst:
            dst.write(lst_celsius, 1)

        print(f"LST saved to: {output_file}")

extreme_heat=[20160807, 20230819]
normal_heat=[20160924, 20230616]
# 输入的栅格文件路径
normal_files = [
    rf'D:\seoul\b_satellite_img\\normal\clipped\clean\clipped_{normal_heat[0]}_B10_cleaned.tif',
    rf'D:\seoul\b_satellite_img\\normal\clipped\clean\clipped_{normal_heat[1]}_B10_cleaned.tif'
]
extreme_files = [
    rf'D:\seoul\b_satellite_img\extreme\clipped\clean\clipped_{extreme_heat[0]}_B10_cleaned.tif',
    rf'D:\seoul\b_satellite_img\extreme\clipped\clean\clipped_{extreme_heat[1]}_B10_cleaned.tif'
]

# 输出目录
output_dir_normal = r'D:\seoul\b_satellite_img\\normal_heat_day_lst'
output_dir_extreme = r'D:\seoul\b_satellite_img\\extreme_heat_day_lst'

# 计算并保存 LST
calculate_lst(normal_files, output_dir_normal)
calculate_lst(extreme_files, output_dir_extreme)

LST saved to: D:\seoul\b_satellite_img\\normal_heat_day_lst\clipped_20160924_B10_cleaned_LST.tif
LST saved to: D:\seoul\b_satellite_img\\normal_heat_day_lst\clipped_20230616_B10_cleaned_LST.tif
LST saved to: D:\seoul\b_satellite_img\\extreme_heat_day_lst\clipped_20160807_B10_cleaned_LST.tif
LST saved to: D:\seoul\b_satellite_img\\extreme_heat_day_lst\clipped_20230819_B10_cleaned_LST.tif


# 相减，然后得到HR map

In [3]:
import rasterio
import numpy as np
import os

def calculate_heat_resilience(normal_files, extreme_files, output_dir, nodata_value=-9999):
    os.makedirs(output_dir, exist_ok=True)

    if len(normal_files) != len(extreme_files):
        raise ValueError("Number of normal and extreme files must be the same.")

    for normal_file, extreme_file in zip(normal_files, extreme_files):
        with rasterio.open(normal_file) as normal_src, rasterio.open(extreme_file) as extreme_src:
            normal_data = normal_src.read(1).astype(float)
            extreme_data = extreme_src.read(1).astype(float)

            if normal_data.shape != extreme_data.shape:
                raise ValueError(f"Dimension mismatch between {normal_file} and {extreme_file}.")

            # 强制处理 nodata
            if normal_src.nodata is not None:
                normal_data[normal_data == normal_src.nodata] = np.nan
            else:
                normal_data[normal_data == nodata_value] = np.nan

            if extreme_src.nodata is not None:
                extreme_data[extreme_data == extreme_src.nodata] = np.nan
            else:
                extreme_data[extreme_data == nodata_value] = np.nan

            # 差值
            heat_resilience = normal_data - extreme_data

            # 打印调试
            print(f"\n📁 Processing: {os.path.basename(normal_file)} vs {os.path.basename(extreme_file)}")
            print("🔹 First 10 values from normal_data (non-NaN):", normal_data[~np.isnan(normal_data)].flatten()[:10])
            print("🔹 First 10 values from extreme_data (non-NaN):", extreme_data[~np.isnan(extreme_data)].flatten()[:10])
            print("🔸 First 10 values from heat_resilience (non-NaN):", heat_resilience[~np.isnan(heat_resilience)].flatten()[:10])

            # 替换 NaN 为 nodata
            heat_resilience[np.isnan(heat_resilience)] = nodata_value

            profile = normal_src.profile
            profile.update({
                'dtype': 'float32',
                'nodata': nodata_value
            })

            year = os.path.basename(normal_file).split('_')[1][:4]
            output_file = os.path.join(output_dir, f"{year}_heat_resilience.tif")

            with rasterio.open(output_file, 'w', **profile) as dst:
                dst.write(heat_resilience.astype('float32'), 1)

            print(f"✅ Heat resilience raster saved to: {output_file}")


extreme_heat=[20160807, 20230819]
normal_heat=[20160924, 20230616]
# 输入的栅格文件路径
normal_files_lst = [
    rf'D:\seoul\b_satellite_img\normal_heat_day_lst\clipped_{normal_heat[0]}_B10_cleaned_lst.tif',
    rf'D:\seoul\b_satellite_img\normal_heat_day_lst\clipped_{normal_heat[1]}_B10_cleaned_lst.tif'
]
extreme_files_lst = [
    rf'D:\seoul\b_satellite_img\extreme_heat_day_lst\clipped_{extreme_heat[0]}_B10_cleaned_lst.tif',
    rf'D:\seoul\b_satellite_img\extreme_heat_day_lst\clipped_{extreme_heat[1]}_B10_cleaned_lst.tif'
]

# 输出目录
output_dir = r'D:\seoul\b_satellite_img\heat_resilience'

# 计算并保存 heat resilience
calculate_heat_resilience(normal_files_lst, extreme_files_lst, output_dir, nodata_value=-9999)


📁 Processing: clipped_20160924_B10_cleaned_lst.tif vs clipped_20160807_B10_cleaned_lst.tif
🔹 First 10 values from normal_data (non-NaN): [23.6862011  24.06218338 23.54606247 23.81266785 23.80583191 24.33562469
 24.08952713 24.38005829 24.65008354 24.62957382]
🔹 First 10 values from extreme_data (non-NaN): [32.88751221 32.70977402 32.58330536 32.68584824 35.6390152  36.1790657
 36.73620224 34.06672668 34.63412094 35.2767067 ]
🔸 First 10 values from heat_resilience (non-NaN): [-6.46005821 -6.33359146 -6.21737671 -6.35751724 -7.13340569 -7.32481766
 -7.54698944 -7.05821037 -7.19835281 -7.35899734]
✅ Heat resilience raster saved to: D:\seoul\b_satellite_img\heat_resilience\2016_heat_resilience.tif

📁 Processing: clipped_20230616_B10_cleaned_lst.tif vs clipped_20230819_B10_cleaned_lst.tif
🔹 First 10 values from normal_data (non-NaN): [26.51290321 27.26828575 26.66671371 27.00509834 26.90255737 27.70921135
 27.27512169 27.61008835 27.86302185 27.91087341]
🔹 First 10 values from extreme_data

# 计算面积和占比

In [5]:
import geopandas as gpd
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
def add_lst_ratio_within_only(clipped_gdf, city_path, output_path):
    # ✅ STEP 1: 读取 grid 文件（即 city_gdf），用其 CRS 作为标准
    city_gdf = gpd.read_file(city_path)
    target_crs = city_gdf.crs  # ⬅️ 使用它为主
    clipped = clipped_gdf.to_crs(target_crs)
    city_gdf = city_gdf.to_crs(target_crs)

    # ✅ STEP 2: 创建 city_id 和 Shape_Area
    city_gdf = city_gdf.reset_index(drop=True)
    city_gdf['city_id'] = city_gdf.index
    # city_gdf['Shape_Area'] = city_gdf.geometry.area
    city_gdf['lst'] = 0.0  # 初始化

    # ✅ STEP 3: 找出每个 grid 内完全包含的 clipped polygon
    for idx, row in city_gdf.iterrows():
        grid_geom = row.geometry
        inside = clipped[clipped.within(grid_geom)]
        if not inside.empty:
            area_sum = inside.geometry.area.sum()
            city_gdf.at[idx, 'lst'] = area_sum

    # ✅ STEP 4: 计算 lst_ratio
    city_gdf['lst_ratio'] = city_gdf['lst'] / city_gdf['Shape_Area']

    # ✅ STEP 5: 导出为 ITRF_2000_UTM_K
    itrf_proj = (
        '+proj=tmerc +lat_0=38 +lon_0=127.5 '
        '+k=0.9996 +x_0=1000000 +y_0=2000000 '
        '+ellps=GRS80 +units=m +no_defs'
    )
    city_gdf_final = city_gdf.to_crs(itrf_proj)
    city_gdf_final.to_file(output_path, driver='ESRI Shapefile')

    print(f"✅ 完全包含统计完成并保存: {output_path}")




# === 设置路径 ===
# 所有 clipped shapefiles（一个 per grid）
clipped_folder = r'D:\seoul\b_satellite_img\shapefiles_clipped_with_fid'
clipped_files = [f for f in os.listdir(clipped_folder) if f.endswith('.shp')]

# 所有 grid shapefiles
grid_folder = r'D:\seoul\grids'
grid_files = [f for f in os.listdir(grid_folder) if f.endswith('.shp')]

# 输出文件夹
output_folder = r'D:\seoul\grids\city_ratio_outputs'
os.makedirs(output_folder, exist_ok=True)

def process_pair(grid_file):
    try:
        grid_name = os.path.splitext(grid_file)[0]
        grid_path = os.path.join(grid_folder, grid_file)

        matched_clipped = [
            f for f in clipped_files if grid_name in f
        ]

        if not matched_clipped:
            print(f"⚠️ 找不到与 {grid_file} 匹配的 clipped 文件")
            return

        clipped_path = os.path.join(clipped_folder, matched_clipped[0])
        clipped_gdf = gpd.read_file(clipped_path)
        clipped_gdf = clipped_gdf.reset_index(drop=True)
        clipped_gdf['city_id'] = clipped_gdf['FID'] if 'FID' in clipped_gdf.columns else clipped_gdf.index

        output_path = os.path.join(output_folder, f'city2016_lst_ratio_{grid_name}.shp')
        if os.path.exists(output_path):
            print(f"⏩ 已存在，跳过: {output_path}")
            return

        add_lst_ratio_within_only(clipped_gdf, grid_path, output_path)

    except Exception as e:
        print(f"❌ 处理 {grid_file} 失败: {e}")

# 🚀 并行处理
print("🚀 开始批量匹配 grid ↔ clipped...")
with ThreadPoolExecutor(max_workers=4) as executor:
    executor.map(process_pair, grid_files)

🚀 开始批量匹配 grid ↔ clipped...
✅ 完全包含统计完成并保存: D:\seoul\grids\city_ratio_outputs\city2016_lst_ratio_grid_1080m.shp
✅ 完全包含统计完成并保存: D:\seoul\grids\city_ratio_outputs\city2016_lst_ratio_grid_450m.shp
✅ 完全包含统计完成并保存: D:\seoul\grids\city_ratio_outputs\city2016_lst_ratio_grid_360m.shp
✅ 完全包含统计完成并保存: D:\seoul\grids\city_ratio_outputs\city2016_lst_ratio_grid_480m.shp
✅ 完全包含统计完成并保存: D:\seoul\grids\city_ratio_outputs\city2016_lst_ratio_grid_600m.shp
✅ 完全包含统计完成并保存: D:\seoul\grids\city_ratio_outputs\city2016_lst_ratio_grid_720m.shp
✅ 完全包含统计完成并保存: D:\seoul\grids\city_ratio_outputs\city2016_lst_ratio_grid_960m.shp
✅ 完全包含统计完成并保存: D:\seoul\grids\city_ratio_outputs\city2016_lst_ratio_grid_840m.shp
✅ 完全包含统计完成并保存: D:\seoul\grids\city_ratio_outputs\city2016_lst_ratio_grid_240m.shp
✅ 完全包含统计完成并保存: D:\seoul\grids\city_ratio_outputs\city2016_lst_ratio_grid_120m.shp


In [8]:
import geopandas as gpd

file = r'D:\seoul\Final_data\building\Raw_data\N1A_B0010000.shp'
data = gpd.read_file(file)
data


,명칭,구분,종류,용도,주기,층수,측량방법,UFID,geometry
0,월곡레미안루나밸리아파트,None,무벽건물,근린생활시설,None,1,사진측량,1000037705062B00110000000000015035,"POLYGON ((203784.871 555598.270, 203781.688 55..."
1,None,None,무벽건물,근린생활시설,None,1,사진측량,1000037705062B00110000000000015034,"POLYGON ((203951.432 553525.396, 203950.690 55..."
2,None,None,무벽건물,근린생활시설,None,1,사진측량,1000037705062B00110000000000015033,"POLYGON ((203391.654 554005.286, 203390.088 55..."
3,None,None,무벽건물,근린생활시설,None,1,사진측량,1000037705062B00110000000000015032,"POLYGON ((203420.382 553624.785, 203420.273 55..."
4,None,None,무벽건물,근린생활시설,None,1,사진측량,1000037705062B00110000000000015031,"POLYGON ((204027.314 552985.679, 204025.494 55..."
...,...,...,...,...,...,...,...,...,...
15025,None,None,주택외건물,근린생활시설,None,1,사진측량,1000037705062B00110000000000015037,"POLYGON ((203561.891 553479.878, 203549.031 55..."
15026,None,None,일반주택,주택,None,1,사진측량,1000037705062B00110000000000015038,"MULTIPOLYGON (((203530.123 555603.747, 203528...."
15027,None,None,주택외건물,의료시설,고려대의료원안암병원,6,사진측량,1000037705062B00110000000000015039,"POLYGON ((202336.340 554229.020, 202331.580 55..."
15028,래미안전농크레시티아파트,None,아파트,주택,102동,12,사진측량,1000037705062B00110000000000015040,"MULTIPOLYGON (((204416.791 552939.959, 204415...."


In [ ]:
print(data["종류"].unique())
print(data["종류"].value_counts())

In [1]:
import geopandas as gpd

file = r'D:\seoul\Final_data\skyviewfactor\svf calculation_qgis\bldg_bd_merge.shp'
data = gpd.read_file(file)
data


,F__,F___1,F___2,F___3,F___4,F___5,F____,UFID,height,area,...,stats_mean,height_asl,BJCD,NAME,DIVI,SCLS,FMTA,Shape_Leng,Shape_Area,geometry
0,????????????,None,????,??????,None,1.0,????,1000037705062B00110000000000015035,3.0,173.10000,...,46.000000,0.0,None,None,None,None,None,66.464533,173.100166,"POLYGON ((203784.871 555598.270, 203781.688 55..."
1,None,None,????,??????,None,1.0,????,1000037705062B00110000000000015034,3.0,4.72697,...,22.000000,0.0,None,None,None,None,None,9.166338,4.726969,"POLYGON ((203951.432 553525.396, 203950.690 55..."
2,None,None,????,??????,None,1.0,????,1000037705062B00110000000000015033,3.0,9.79550,...,27.000000,0.0,None,None,None,None,None,16.017146,9.795502,"POLYGON ((203391.654 554005.286, 203390.088 55..."
3,None,None,????,??????,None,1.0,????,1000037705062B00110000000000015032,3.0,7.32160,...,27.000000,0.0,None,None,None,None,None,11.519522,7.321605,"POLYGON ((203420.382 553624.785, 203420.273 55..."
4,None,None,????,??????,None,1.0,????,1000037705062B00110000000000015031,3.0,5.57539,...,22.000000,0.0,None,None,None,None,None,10.341786,5.575397,"POLYGON ((204027.314 552985.679, 204025.494 55..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
800575,None,None,????,?????????,????,1.0,????,1000037709024B00110000000000000030,3.0,3.63904,...,115.919530,0.0,None,None,None,None,None,10.804301,3.639043,"POLYGON ((207117.559 538956.350, 207116.942 53..."
800576,None,None,????,????,None,1.0,????,1000037709024B00110000000000000031,3.0,135.66600,...,114.264783,0.0,None,None,None,None,None,45.951656,135.666603,"POLYGON ((206977.970 538944.683, 206974.952 53..."
800577,None,None,????,????,None,1.0,????,1000037709024B00110000000000000032,3.0,13.48380,...,109.000000,0.0,None,None,None,None,None,14.793161,13.483810,"POLYGON ((206690.450 538780.768, 206688.846 53..."
800578,None,None,??,????,None,1.0,????,1000037709024B00110000000000000872,3.0,81.23440,...,105.865205,0.0,None,None,None,None,None,38.382723,81.234488,"POLYGON ((206699.007 538736.290, 206686.887 53..."


In [2]:
print(data["F___2"].unique())
print(data["F___2"].value_counts())

['????' '?????' '???' '??']
F___2
????     514541
?????    245994
???       38547
??         1498
Name: count, dtype: int64


In [ ]:
# 무벽건물
# 가건물

In [13]:
import geopandas as gpd

file = r'D:\seoul\c_data\a_building\AL_11_D010_20160806.shp'

data_small = gpd.read_file(file, encoding='euc_kr')#, rows=50000
data_small

,A0,A1,A2,A3,A4,A5,A6,A7,A8,A9,...,A14,A15,A16,A17,A18,A19,A20,A21,A22,geometry
0,24937,1988197105154543659600000000,1111010100100010000,1111010100,서울특별시 종로구 청운동,1,1,일반,02000,공동주택,...,4686.15,0.0,0.0,0.0,0.0,2381,0,B00100000000TT1VW,1899-12-30,"POLYGON ((197119.187 454372.508, 197103.397 45..."
1,24922,1988197172864542613800000000,1111010100100010000,1111010100,서울특별시 종로구 청운동,1,1,일반,02000,공동주택,...,6572.10,0.0,0.0,0.0,0.0,2382,0,B00100000000TT1GH,1899-12-30,"POLYGON ((197167.005 454248.176, 197144.745 45..."
2,24935,1988197056224543775000000000,1111010100100010000,1111010100,서울특별시 종로구 청운동,1,1,일반,02000,공동주택,...,5489.43,0.0,0.0,0.0,0.0,2383,0,B00100000000TT1TU,1899-12-30,"POLYGON ((197077.797 454396.579, 197045.366 45..."
3,24936,1988197076494543686200000000,1111010100100010000,1111010100,서울특별시 종로구 청운동,1,1,일반,02000,공동주택,...,4772.55,0.0,0.0,0.0,0.0,2384,0,B00100000000TT1UV,1899-12-30,"POLYGON ((197092.447 454380.448, 197070.876 45..."
4,24958,1988197149274543345500000000,1111010100100010000,1111010100,서울특별시 종로구 청운동,1,1,일반,02000,공동주택,...,6732.21,0.0,0.0,0.0,0.0,2385,0,B00100000000TT2GI,1899-12-30,"POLYGON ((197183.247 454318.456, 197174.476 45..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
746183,5613,0000215570844507756700000000,1174011000200080011,1174011000,서울특별시 강동구 강일동,8-11,2,산,None,None,...,0.00,0.0,0.0,0.0,0.0,None,None,B00100000000WP4CF,1899-12-30,"POLYGON ((215570.446 450772.851, 215569.000 45..."
746184,5523,0000215740974502818900000000,1174011000200240001,1174011000,서울특별시 강동구 강일동,24-1,2,산,None,None,...,0.00,0.0,0.0,0.0,0.0,None,None,B00100000000WOHET,1899-12-30,"POLYGON ((215737.971 450288.975, 215749.246 45..."
746185,5518,0000215654324502800000000000,1174011000200240001,1174011000,서울특별시 강동구 강일동,24-1,2,산,None,None,...,0.00,0.0,0.0,0.0,0.0,None,None,B00100000000WOHFU,1899-12-30,"POLYGON ((215649.260 450282.765, 215659.061 45..."
746186,5519,0000215654644502750400000000,1174011000200240001,1174011000,서울특별시 강동구 강일동,24-1,2,산,None,None,...,0.00,0.0,0.0,0.0,0.0,None,None,B00100000000WPCKV,1899-12-30,"POLYGON ((215659.612 450273.575, 215649.804 45..."


In [ ]:
print(data_small["A8"].unique())
print(data_small["A9"].unique())
print(data_small["A16"].value_counts())

['公寓' 无 '文化及集会设施' '独栋住宅' '老年人设施' '宗教设施' '一级社区生活设施' '二级社区生活设施'
'体育设施' '教育及研究设施' '教育及研究福利设施' '汽车相关设施' '商业设施' '社区生活设施' '工厂' '危险品储存及处理设施'
'仓储设施' '住宿设施' '医疗设施' '旅游及休息设施' '公共设施' '休闲设施' '销售设施' '培训设施' '动植物相关设施'
'交通设施' '销售及商业设施' '广播及通信设施' '污水及废物处理设施' '惩教及军事设施' '电力发电设施’‘墓地相关设施’‘殡仪馆’]

In [17]:
num_none = data_small["A8"].isnull().sum()
print("值为 None 或 NaN 的行数：", num_none)

值为 None 或 NaN 的行数： 146425


In [11]:
import geopandas as gpd

file = r'D:\seoul\c_data\a_building\\AL_D010_11_20240607.shp'

data_small = gpd.read_file(file, encoding='euc_kr')#, rows=50000
data_small

,A0,A1,A2,A3,A4,A5,A6,A7,A8,A9,...,A20,A21,A22,A23,A24,A25,A26,A27,A28,geometry
0,32,1991201839054527769900000000,1111017500107040000,1111017500,서울특별시 종로구 숭인동,704,1,일반,01000,단독주택,...,N,B00100000000T30Z9,2024-06-04,11110,None,None,3.0,0.0,2017-05-30,"POLYGON ((201913.638 553085.323, 201912.228 55..."
1,34,1967201343844527773800000000,1111017500100560024,1111017500,서울특별시 종로구 숭인동,56-24,1,일반,01000,단독주택,...,N,B00100000000T311B,2024-06-04,11110,None,None,3.0,0.0,2018-06-19,"POLYGON ((201413.758 553078.463, 201410.198 55..."
2,36,1962201375194527742000000000,1111017500100560054,1111017500,서울특별시 종로구 숭인동,56-54,1,일반,01000,단독주택,...,N,B00100000000T313D,2024-06-04,11110,None,None,2.0,0.0,2017-02-24,"POLYGON ((201442.598 553075.703, 201442.298 55..."
3,37,1979201388484527710100000000,1111017500100560053,1111017500,서울특별시 종로구 숭인동,56-53,1,일반,01000,단독주택,...,N,B00100000000T314E,2024-06-04,11110,None,None,2.0,1.0,2018-11-06,"POLYGON ((201458.568 553069.403, 201453.548 55..."
4,38,0000200468294527764000000000,1111016500100280011,1111016500,서울특별시 종로구 이화동,28-11,1,일반,None,None,...,None,B00100000000T315F,2024-06-04,11110,None,None,0.0,0.0,None,"POLYGON ((200540.538 553080.543, 200536.438 55..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
696062,10105,1990198254284494774700000000,1117010100102670002,1117010100,서울특별시 용산구 후암동,267-2,1,일반,01000,단독주택,...,Y,B001000000013AU3O,2024-06-04,11170,None,None,2.0,1.0,2024-05-23,"POLYGON ((198316.762 549783.247, 198323.391 54..."
696063,8522,1995198146964497771000000000,1117010100100480012,1117010100,서울특별시 용산구 후암동,48-12,1,일반,01000,단독주택,...,Y,B00100000000RMMAN,2024-06-04,11170,None,None,2.0,1.0,2024-05-23,"POLYGON ((198212.062 550079.067, 198212.682 55..."
696064,12847,2003201075314427046800000000,1165010800115930007,1165010800,서울특별시 서초구 서초동,1593-7,1,일반,02000,공동주택,...,N,B001000000012Z3TC,2024-06-04,11650,서초 이오빌,None,24.0,6.0,2024-05-23,"POLYGON ((201133.460 543032.394, 201136.678 54..."
696065,9841,1993205799614483901100000000,1121510500108510019,1121510500,서울특별시 광진구 자양동,851-19,1,일반,03000,제1종근린생활시설,...,N,B00100000000W3WOX,2024-06-04,11215,None,None,5.0,1.0,2024-05-23,"POLYGON ((205871.117 548687.488, 205862.148 54..."


In [12]:
print(data_small["A8"].unique())
print(data_small["A9"].unique())

['01000' None '04000' '02000' '03000' '14000' '10000' '11000' '17000'
 '06000' '15000' '05000' '19000' '20000' '18000' '07000' '09000' 'Z8000'
 '16000' '27000' 'Z3000' '13000' 'Z9000' 'Z5000' '21000' '12000' 'Z6000'
 '08000' '22000' '24000' '23000' '25000' '01003' '26000' '31000' '29000'
 '30000' '28000']
['단독주택' None '제2종근린생활시설' '공동주택' '제1종근린생활시설' '업무시설' '교육연구시설' '노유자시설' '공장'
 '종교시설' '숙박시설' '문화및집회시설' '위험물저장및처리시설' '자동차관련시설' '창고시설' '판매시설' '의료시설'
 '교육연구및복지시설' '위락시설' '관광휴게시설' '근린생활시설' '운동시설' '공공용시설' '동.식물 관련시설' '수련시설'
 '판매및영업시설' '운수시설' '분뇨.쓰레기처리시설' '방송통신시설' '교정및군사시설' '발전시설' '다가구주택' '묘지관련시설'
 '장례식장' '가설건축물']


# grid map

In [ ]:
import os
import rasterio
import numpy as np
import geopandas as gpd
from rasterio.mask import mask
from shapely.geometry import mapping

def calculate_polygon_means(raster_path, vector_path):
    """
    根据矢量多边形计算栅格的平均值。
    :param raster_path: 栅格文件路径。
    :param vector_path: 矢量文件路径。
    :return: 列名（基于日期）和对应的多边形平均值列表。
    """
    try:
        # 提取日期部分作为列名
        filename = os.path.basename(raster_path)
        date_str = filename.split('_')[2]  # 提取类似 '20170730' 的部分

        # 打开矢量文件
        vector = gpd.read_file(vector_path)

        # 打开栅格文件
        with rasterio.open(raster_path) as raster:
            raster_crs = raster.crs

            # 矢量文件需要投影到栅格的 CRS
            if vector.crs != raster_crs:
                vector = vector.to_crs(raster_crs)
                print(f"Reprojected vector CRS to match raster CRS.")

            # 用于存储每个多边形的平均值
            mean_values = []

            # 遍历每个多边形
            for idx, row in vector.iterrows():
                geom = row['geometry']
                # 使用多边形裁剪栅格
                out_image, out_transform = mask(raster, [mapping(geom)], crop=True, all_touched=True)


                # 如果裁剪区域无数据，赋值为 NaN
                if out_image.size == 0:
                    mean_values.append(np.nan)
                else:
                    data = out_image[0]  # 获取第一波段
                    # 计算平均值，忽略无效值（如 0 或 NaN）
                    mean_val = np.nanmean(data[data != raster.nodata])
                    mean_values.append(mean_val)

            return date_str, mean_values

    except Exception as e:
        print(f"Error processing file {raster_path}: {e}")
        return None, []

# 主函数：处理多个栅格文件并保存到单个文件
def process_rasters_to_single_file(raster_files, vector_path, output_path, group_name, year):
    """
    处理多个栅格文件，将结果添加为矢量文件的新列。
    :param raster_files: 栅格文件路径列表。
    :param vector_path: 矢量文件路径。
    :param output_path: 输出矢量文件路径。
    :param group_name: 当前组的名称（如 normal、extreme 等）。
    :param year: 对应的年份列表。
    """
    try:
        # 打开矢量文件
        vector = gpd.read_file(vector_path)

        # 遍历栅格文件并计算每个多边形的平均值
        for idx, raster_file in enumerate(raster_files):
            date_str, mean_values = calculate_polygon_means(raster_file, vector_path)
            if date_str and mean_values:
                # 添加新列到矢量文件，列名为 group_name_年份
                column_name = f"{group_name}_{year[idx]}"
                vector[column_name] = mean_values
                print(f"Added column: {column_name}")

        # 保存更新后的矢量文件
        vector.to_file(output_path)
        print(f"Saved updated vector file to: {output_path}")

    except Exception as e:
        print(f"Error processing rasters for group {group_name}: {e}")


# 主脚本
input_folder = r'D:\seoul\grids\city_ratio_outputs'
output_folder = r'D:\seoul\grids\lst_map'
os.makedirs(output_folder, exist_ok=True)

year = '2016'
# 输入的栅格文件路径
normal_files = [
    r'D:\seoul\b_satellite_img\normal_heat_day_lst\clipped_20160924_B10_cleaned_LST.tif'
]
extreme_files = [
    r'D:\seoul\b_satellite_img\extreme_heat_day_lst\clipped_20160807_B10_cleaned_LST.tif'
]
hr_files = [
    rf'D:\seoul\Final_data\landsat_final\data\heat_resilience\{year}_heat_resilience.tif'
]

# 文件列表与名称的映射
file_groups = [normal_files, extreme_files, hr_files]
group_names = ['nor', 'ext', 'hr']
years =[[year],[year],[year]]

for filename in os.listdir(input_folder):
    if filename.endswith('.shp'):
        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, filename)

        # 初始化矢量文件
        final_vector = gpd.read_file(input_path)

        # 循环处理文件
        for i in range(len(file_groups)):
            group_files = file_groups[i]  # 当前组的文件列表
            group_name = group_names[i]  # 当前组的名称
            print(f"Processing {group_name} files...")  # 可选：打印当前处理的文件组
            for idx, raster_file in enumerate(group_files):
                # 获取单个栅格文件的平均值
                date_str, mean_values = calculate_polygon_means(raster_file, input_path)
                if date_str and mean_values:
                    column_name = f"{group_name}_{years[i][idx]}"
                    final_vector[column_name] = mean_values
                    print(f"Added column: {column_name}")
        if os.path.exists(output_path):
            print(f"⏩ 已存在，跳过: {output_path}")
            continue
        # 保存最终结果
        final_vector.to_file(output_path)
        print(f"Saved final vector file to: {output_path}")